In [0]:
!pip install -U \
    category-encoders \
    # cffi==1.16.0 \
    cloudpickle==3.0.0 \
    nltk==3.9.2 \
    defusedxml==0.7.1 \
    graphviz==0.20.3 \
    holidays==0.54 \
    lightgbm==4.5.0 \
    # lz4==4.3.3 \
    matplotlib==3.9.2 \
    psutil==5.9.8 \
    pyarrow==15.0.2 \
    optuna \
    sentence-transformers

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install sentence-transformers

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install emoji nltk keybert

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
!pip install hdbscan umap

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
train_df = pd.read_csv(
    "/Workspace/Users/saurabh.prajapati@tvsmotor.com/Project/data/raw/train.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)


In [0]:
import pandas as pd
test_df = pd.read_csv(
    "/Workspace/Users/saurabh.prajapati@tvsmotor.com/Project/data/raw/test.tsv",
    sep='\t',
    names=[
        "customer_identifier",
        "medicine_name",
        "rating",
        "effectiveness",
        "side_effects",
        "illness",
        "review_benefits",
        "review_sideEffects",
        "review_overall"
    ],
    header=None
)

In [0]:
df=pd.concat([train_df,test_df],axis=0)

In [0]:
df.isnull().sum()

customer_identifier     0
medicine_name           0
rating                  0
effectiveness           0
side_effects            0
illness                 1
review_benefits        23
review_sideEffects     98
review_overall         13
dtype: int64

In [0]:
mode_illness = df['illness'].mode()[0]
df['illness'] = df['illness'].fillna(mode_illness)
df['review_benefits'] = df['review_benefits'].fillna("")
df['review_sideEffects'] = df['review_sideEffects'].fillna("")
df['review_overall'] = df['review_overall'].fillna("")
df['rating']=df['rating'].astype(str)

In [0]:
review_template = (
    "review_benefits: {review_benefits}\n"
    "review_sideEffects: {review_sideEffects}\n"
    "review_overall: {review_overall}  \n"
    "medicine_name: {medicine_name}  \n"
    "effectiveness: {effectiveness}  \n"
    "side_effects: {side_effects}  \n"

)

df['combined_feats'] = df.apply(lambda row: review_template.format(**row), axis=1)


In [0]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO')
text_embeddings = model.encode(df["combined_feats"].tolist(), show_progress_bar=True)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/130 [00:00<?, ?it/s]

In [0]:
df.shape

(4143, 10)

In [0]:
# Convert the 2D embeddings array to a list of 1D arrays (or lists)
df['embeddings'] = [row for row in text_embeddings]

In [0]:
from sklearn.metrics.pairwise   import cosine_similarity

def recommend_medicine_by_symptom(symptom, top_n=2):
    user_embedding = model.encode([symptom])[0]
    similarities = cosine_similarity([user_embedding], text_embeddings)[0]
    temp_df = df.copy()
    temp_df['cosine_similarity'] = similarities
    recommendations = temp_df.sort_values(by='cosine_similarity', ascending=False)["medicine_name"].head(top_n)
    return recommendations

# Example usage:


In [0]:
recommend_medicine_by_symptom("I’m looking for something that can help reduce acne scars and improve overall skin texture. My pores are large and skin looks dull.", top_n=2)

2448            avita
931     retin-a-micro
Name: medicine_name, dtype: object